<a href="https://colab.research.google.com/github/SANTHOSH-C08/AI-INTERNSHIP/blob/main/Resume_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install faiss-cpu sentence_transformers pypdf

In [19]:
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [20]:
model= SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
resume_files = [
    ("Santhosh_C_Resume.pdf", "Santhosh C"),
    ("Alaguraja_Resume.pdf", "Akaguraja K"),
    ("Shivaji_Resume.pdf", "Shivaji S B")
]

In [26]:
def extract_text(pdf_path):
  reader = PdfReader(pdf_path)

  text = ""
  for page in reader.pages:
    page_text = page.extract_text()

    if page_text:
      text += page_text + "\n"
  return text

In [27]:
def chunk_text(text, chunk_size=500):
  chunks=[]
  for i in range(0,len(text),chunk_size):
    chunks.append(text[i:i+chunk_size])
  return chunks

In [28]:
all_chunks = []
metadata = []
for pdf_file, candidate in resume_files:
  text = extract_text(pdf_file)
  chunks = chunk_text(text)

  for chunk in chunks:
    all_chunks.append(chunk)
    metadata.append({
        "candidate": candidate,
        "file": pdf_file
    })

In [30]:
embeddings = model.encode(all_chunks)

In [31]:
embeddings.shape

(17, 384)

In [32]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype("float32"))
print("Total chunks stored:", index.ntotal)

Total chunks stored: 17


In [35]:
def search_resumes(query, top_k=3):
  query_embeddings = model.encode([query])
  distances,indices = index.search(
      np.array(query_embeddings).astype("float32"),
      top_k
  )

  results = []

  for idx in indices[0]:

      results.append({
            "candidate": metadata[idx]["candidate"],
            "file": metadata[idx]["file"],
            "chunk": all_chunks[idx]
      })

  return results


In [36]:
query = "Who knows excel?"

results = search_resumes(query)

In [37]:
for result in results:

    print("-"*50)

    print("Candidate :", result["candidate"])

    print("File      :", result["file"])

    print("Content   :")

    print(result["chunk"])

--------------------------------------------------
Candidate : Akaguraja K
File      : Alaguraja_Resume.pdf
Content   :
ALAGURAJA K
(+91) 7010744391  |  alaguraja698900@gmail.com  |  github.com/Alaguraja787  |  linkedin.com/in/alagu-raja-86b23632a
SUMMARY
Third-year B.Tech Artificial Intelligence & Data Science student with hands-on experience in Python, machine learning, and web 
technologies. Passionate about data science, software development, and problem-solving. Seeking an opportunity to apply technical 
expertise and grow as a professional within a forward-thinking organisation.
TECHNICAL SKILLS
Programming
--------------------------------------------------
Candidate : Santhosh C
File      : Santhosh_C_Resume.pdf
Content   :
l Project 
• Designed and developed a habit tracking application to help users monitor and build consistent daily 
routines. 
• Implemented a clean, user-friendly interface focusing on frontend design principles and usability. 
• Practiced full project lifecy